### 本notebook进行联邦学习（Federated Learning）

在联邦学习中，每个客户只能访问自己的私有数据集，但他们协作训练一个模型，该模型在任何其他客户的数据上都能取得满意的结果。

In [4]:
import sys
import os

from pathlib import Path

parent = Path(os.path.abspath("")).resolve().parents[0]
if parent not in sys.path:
    sys.path.insert(0, str(parent))

In [5]:
import copy

import random

from collections import OrderedDict

import numpy as np
import torch

from argparse import Namespace

In [6]:
from ml.utils.data_utils import read_data, generate_time_lags, time_to_feature, handle_nans, to_Xy, \
    to_torch_dataset, to_timeseries_rep, assign_statistics, \
    to_train_val, scale_features, get_data_by_area, remove_identifiers, get_exogenous_data_by_area, handle_outliers

In [7]:
from ml.utils.train_utils import train, test

In [8]:
from ml.models.mlp import MLP
from ml.models.rnn import RNN
from ml.models.lstm import LSTM
from ml.models.gru import GRU
from ml.models.cnn import CNN
from ml.models.rnn_autoencoder import DualAttentionAutoEncoder

In [9]:
from ml.fl.defaults import create_regression_client
from ml.fl.client_proxy import SimpleClientProxy
from ml.fl.server.server import Server

In [10]:
args = Namespace(
    data_path='../CBL-dataset/LCL-June2015v2_0.csv', # 电力数据集路径
    data_path_test=None, # 测试数据集（可选）
    test_size=0.2, # 验证集比例
    targets=['kwh'], # 目标列：用电量
    num_lags=10, # 用于输入的过去观测值数量

    identifier='customer_id', # 标识客户的列名

    nan_constant=0, # 用于转换NaN值的常数
    x_scaler='minmax', # X特征缩放器
    y_scaler='minmax', # y目标缩放器
    outlier_detection=None, # 是否执行异常值处理（flooring and capping）

    
    criterion='mse', # 优化准则，mse或l1
    fl_rounds=10, # 联邦学习轮数
    fraction=1., # 用于随机选择的可用客户百分比
    aggregation="fedavg", # 联邦聚合算法
    epochs=3, # 最大本地训练轮数
    lr=0.001, # 学习率
    optimizer='adam', # 优化器，可以是sgd或adam
    batch_size=128, # 批次大小
    local_early_stopping=False, # 是否使用早停
    local_patience=50, # 早停的耐心值（如果指定）
    max_grad_norm=0.0, # 是否裁剪梯度范数
    reg1=0.0, # l1正则化
    reg2=0.0, # l2正则化

    cuda=True, # 是否使用GPU
    
    seed=0, # 随机种子（可重现性）

    assign_stats=None, # 是否使用统计量作为外生数据, ["mean", "median", "std", "variance", "kurtosis", "skew"]
    use_time_features=False # 是否使用日期时间特征
)

In [11]:
print(f"Script arguments: {args}\n")

Script arguments: Namespace(data_path='../CBL-dataset/LCL-June2015v2_0.csv', data_path_test=None, test_size=0.2, targets=['kwh'], num_lags=10, identifier='customer_id', nan_constant=0, x_scaler='minmax', y_scaler='minmax', outlier_detection=None, criterion='mse', fl_rounds=10, fraction=1.0, aggregation='fedavg', epochs=3, lr=0.001, optimizer='adam', batch_size=128, local_early_stopping=False, local_patience=50, max_grad_norm=0.0, reg1=0.0, reg2=0.0, cuda=True, seed=0, assign_stats=None, use_time_features=False)



In [12]:
device = "cuda" if args.cuda and torch.cuda.is_available() else "cpu"
print(f"Using {device}")

Using cuda


In [13]:
# 异常值检测配置（如果需要）
if args.outlier_detection is not None:
    outlier_columns = ['kwh']  # 电力数据只有kwh列
    # 可以根据不同客户设置不同的异常值处理参数
    # 这里使用统一的参数，也可以为每个客户单独设置
    outlier_kwargs = {}  # 如果需要对特定客户设置，可以添加如 {"MAC000002": (10, 90)}
    args.outlier_columns = outlier_columns
    args.outlier_kwargs = outlier_kwargs

In [14]:
def seed_all():
    # ensure reproducibility
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [15]:
seed_all()

### 预处理方法几乎等同于集中式学习。唯一的区别是缩放操作在每个客户上单独执行。相比之下，在集中式学习中，缩放是通过考虑所有客户的组合数据来执行的。

In [16]:
def make_preprocessing():
    """预处理电力数据集"""
    # 读取数据（使用自定义函数加载电力数据）
    import pandas as pd
    
    # 加载电力数据集
    df = pd.read_csv(args.data_path)
    
    # 清理列名
    df.columns = df.columns.str.strip()
    
    # 重命名列
    df = df.rename(columns={
        'LCLid': 'customer_id',
        'stdorToU': 'tariff_type',
        'DateTime': 'datetime',
        'KWH/hh (per half hour)': 'kwh'
    })
    
    # 转换数据类型
    df['datetime'] = pd.to_datetime(df['datetime'], format="%Y-%m-%d %H:%M:%S.%f", errors='coerce')
    df['kwh'] = pd.to_numeric(df['kwh'], errors='coerce')
    
    # 删除tariff_type列（字符串类型，不适合建模）
    df = df.drop(['tariff_type'], axis=1)
    
    # 设置时间索引
    df.set_index(pd.DatetimeIndex(df["datetime"]), inplace=True)
    df.drop(["datetime"], axis=1, inplace=True)
    
    # 只保留customer_id和kwh列
    df = df[['customer_id', 'kwh']].copy()
    df['kwh'] = df['kwh'].astype("float32")
    
    # 处理缺失值
    df = handle_nans(train_data=df, constant=args.nan_constant,
                     identifier=args.identifier)
    
    # 分割训练/验证集（传递正确的identifier参数）
    train_data, val_data = to_train_val(df, train_size=1-args.test_size, identifier=args.identifier)
    
    # 处理异常值（如果指定）
    if args.outlier_detection is not None:
        train_data = handle_outliers(df=train_data, columns=args.outlier_columns,
                                     identifier=args.identifier, kwargs=args.outlier_kwargs)
    
    # 获取X和y（传递正确的identifier参数）
    X_train, X_val, y_train, y_val = to_Xy(train_data=train_data, val_data=val_data,
                                          targets=args.targets, identifier=args.identifier)
    
    # 缩放X（每个客户单独缩放）
    X_train, X_val, x_scalers = scale_features(train_data=X_train, val_data=X_val,
                                              scaler=args.x_scaler,
                                              per_area=True, # 特征在每个客户上单独缩放
                                              identifier=args.identifier)
    # 缩放y（每个客户单独缩放）
    y_train, y_val, y_scalers = scale_features(train_data=y_train, val_data=y_val,
                                              scaler=args.y_scaler, 
                                              per_area=True,
                                              identifier=args.identifier)
    
    # 生成时间滞后特征
    X_train = generate_time_lags(X_train, args.num_lags, identifier=args.identifier)
    X_val = generate_time_lags(X_val, args.num_lags, identifier=args.identifier)
    y_train = generate_time_lags(y_train, args.num_lags, identifier=args.identifier, is_y=True)
    y_val = generate_time_lags(y_val, args.num_lags, identifier=args.identifier, is_y=True)
    
    # 获取日期时间特征作为外生数据
    date_time_df_train = time_to_feature(
        X_train, args.use_time_features, identifier=args.identifier
    )
    date_time_df_val = time_to_feature(
        X_val, args.use_time_features, identifier=args.identifier
    )
    
    # 获取统计量作为外生数据
    stats_df_train = assign_statistics(X_train, args.assign_stats, args.num_lags,
                                       targets=args.targets, identifier=args.identifier)
    stats_df_val = assign_statistics(X_val, args.assign_stats, args.num_lags, 
                                       targets=args.targets, identifier=args.identifier)
    
    # 合并外生特征（如果有）
    if date_time_df_train is not None or stats_df_train is not None:
        exogenous_data_train = pd.concat([date_time_df_train, stats_df_train], axis=1)
        # 移除重复列（如果有）
        exogenous_data_train = exogenous_data_train.loc[:, ~exogenous_data_train.columns.duplicated()].copy()
        assert len(exogenous_data_train) == len(X_train) == len(y_train)
    else:
        exogenous_data_train = None
    if date_time_df_val is not None or stats_df_val is not None:
        exogenous_data_val = pd.concat([date_time_df_val, stats_df_val], axis=1)
        exogenous_data_val = exogenous_data_val.loc[:, ~exogenous_data_val.columns.duplicated()].copy()
        assert len(exogenous_data_val) == len(X_val) == len(y_val)
    else:
        exogenous_data_val = None
        
    return X_train, X_val, y_train, y_val, exogenous_data_train, exogenous_data_val, x_scalers, y_scalers

In [17]:
X_train, X_val, y_train, y_val, exogenous_data_train, exogenous_data_val, x_scalers, y_scalers = make_preprocessing()

INFO logger 2026-01-07 12:25:19,244 | data_utils.py:383 | Observations info in MAC000002
INFO logger 2026-01-07 12:25:19,246 | data_utils.py:384 | 	Total number of samples:  24158
INFO logger 2026-01-07 12:25:19,247 | data_utils.py:385 | 	Number of samples for training: 19327
INFO logger 2026-01-07 12:25:19,248 | data_utils.py:386 | 	Number of samples for validation:  4831
INFO logger 2026-01-07 12:25:19,317 | data_utils.py:383 | Observations info in MAC000003
INFO logger 2026-01-07 12:25:19,318 | data_utils.py:384 | 	Total number of samples:  35469
INFO logger 2026-01-07 12:25:19,319 | data_utils.py:385 | 	Number of samples for training: 28376
INFO logger 2026-01-07 12:25:19,320 | data_utils.py:386 | 	Number of samples for validation:  7093
INFO logger 2026-01-07 12:25:19,390 | data_utils.py:383 | Observations info in MAC000004
INFO logger 2026-01-07 12:25:19,391 | data_utils.py:384 | 	Total number of samples:  31677
INFO logger 2026-01-07 12:25:19,392 | data_utils.py:385 | 	Number of

In [18]:
X_train.head()

,kwh_lag-10,kwh_lag-9,kwh_lag-8,kwh_lag-7,kwh_lag-6,kwh_lag-5,kwh_lag-4,kwh_lag-3,kwh_lag-2,kwh_lag-1,customer_id
datetime,,,,,,,,,,,
2012-10-12 05:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002
2012-10-12 06:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002
2012-10-12 06:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002
2012-10-12 07:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002
2012-10-12 07:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002


In [19]:
y_train.head()

,kwh,customer_id
datetime,,
2012-10-12 05:30:00,0.0,MAC000002
2012-10-12 06:00:00,0.0,MAC000002
2012-10-12 06:30:00,0.0,MAC000002
2012-10-12 07:00:00,0.0,MAC000002
2012-10-12 07:30:00,0.0,MAC000002


In [20]:
x_scalers, y_scalers

({'MAC000002': MinMaxScaler(),
  'MAC000003': MinMaxScaler(),
  'MAC000004': MinMaxScaler(),
  'MAC000006': MinMaxScaler(),
  'MAC000007': MinMaxScaler(),
  'MAC000008': MinMaxScaler(),
  'MAC000009': MinMaxScaler(),
  'MAC000010': MinMaxScaler(),
  'MAC000011': MinMaxScaler(),
  'MAC000012': MinMaxScaler(),
  'MAC000013': MinMaxScaler(),
  'MAC000016': MinMaxScaler(),
  'MAC000018': MinMaxScaler(),
  'MAC000019': MinMaxScaler(),
  'MAC000020': MinMaxScaler(),
  'MAC000021': MinMaxScaler(),
  'MAC000022': MinMaxScaler(),
  'MAC000023': MinMaxScaler(),
  'MAC000024': MinMaxScaler(),
  'MAC000025': MinMaxScaler(),
  'MAC000026': MinMaxScaler(),
  'MAC000027': MinMaxScaler(),
  'MAC000028': MinMaxScaler(),
  'MAC000029': MinMaxScaler(),
  'MAC000030': MinMaxScaler(),
  'MAC000032': MinMaxScaler(),
  'MAC000033': MinMaxScaler(),
  'MAC000034': MinMaxScaler(),
  'MAC000035': MinMaxScaler(),
  'MAC000036': MinMaxScaler()},
 {'MAC000002': MinMaxScaler(),
  'MAC000003': MinMaxScaler(),
  'MAC0

### 后处理方式与集中式学习相同

In [21]:
def make_postprocessing(X_train, X_val, y_train, y_val, exogenous_data_train, exogenous_data_val, x_scalers, y_scalers):
    """Make data ready to be fed into ml algorithms"""
    # if there are more than one specified areas, get the data per area
    if X_train[args.identifier].nunique() != 1:
        area_X_train, area_X_val, area_y_train, area_y_val = get_data_by_area(X_train, X_val,
                                                                              y_train, y_val, 
                                                                              identifier=args.identifier)
    else:
        area_X_train, area_X_val, area_y_train, area_y_val = None, None, None, None

    # Get the exogenous data per area.
    if exogenous_data_train is not None:
        exogenous_data_train, exogenous_data_val = get_exogenous_data_by_area(exogenous_data_train,
                                                                              exogenous_data_val)
    # transform to np
    if area_X_train is not None:
        for area in area_X_train:
            tmp_X_train, tmp_y_train, tmp_X_val, tmp_y_val = remove_identifiers(
                area_X_train[area], area_y_train[area], area_X_val[area], area_y_val[area],
                identifier=args.identifier)
            tmp_X_train, tmp_y_train = tmp_X_train.to_numpy(), tmp_y_train.to_numpy()
            tmp_X_val, tmp_y_val = tmp_X_val.to_numpy(), tmp_y_val.to_numpy()
            area_X_train[area] = tmp_X_train
            area_X_val[area] = tmp_X_val
            area_y_train[area] = tmp_y_train
            area_y_val[area] = tmp_y_val
    
    if exogenous_data_train is not None:
        for area in exogenous_data_train:
            exogenous_data_train[area] = exogenous_data_train[area].to_numpy()
            exogenous_data_val[area] = exogenous_data_val[area].to_numpy()
    
    # remove identifiers from features, targets
    X_train, y_train, X_val, y_val = remove_identifiers(X_train, y_train, X_val, y_val,
                                                         identifier=args.identifier)
    assert len(X_train.columns) == len(X_val.columns)
    
    num_features = len(X_train.columns) // args.num_lags
    
    # to timeseries representation
    X_train = to_timeseries_rep(X_train.to_numpy(), num_lags=args.num_lags,
                                            num_features=num_features)
    X_val = to_timeseries_rep(X_val.to_numpy(), num_lags=args.num_lags,
                                          num_features=num_features)
    
    if area_X_train is not None:
        area_X_train = to_timeseries_rep(area_X_train, num_lags=args.num_lags,
                                                     num_features=num_features)
        area_X_val = to_timeseries_rep(area_X_val, num_lags=args.num_lags,
                                                   num_features=num_features)
    
    # transform targets to numpy
    y_train, y_val = y_train.to_numpy(), y_val.to_numpy()
    
    if exogenous_data_train is not None:
        exogenous_data_train_combined, exogenous_data_val_combined = [], []
        for area in exogenous_data_train:
            exogenous_data_train_combined.extend(exogenous_data_train[area])
            exogenous_data_val_combined.extend(exogenous_data_val[area])
        exogenous_data_train_combined = np.stack(exogenous_data_train_combined)
        exogenous_data_val_combined = np.stack(exogenous_data_val_combined)
        exogenous_data_train["all"] = exogenous_data_train_combined
        exogenous_data_val["all"] = exogenous_data_val_combined
    return X_train, X_val, y_train, y_val, area_X_train, area_X_val, area_y_train, area_y_val, exogenous_data_train, exogenous_data_val

In [22]:
X_train, X_val, y_train, y_val, client_X_train, client_X_val, client_y_train, client_y_val, exogenous_data_train, exogenous_data_val = make_postprocessing(X_train, X_val, y_train, y_val, exogenous_data_train, exogenous_data_val, x_scalers, y_scalers)

In [23]:
client_X_train.keys()

dict_keys(['MAC000002', 'MAC000003', 'MAC000004', 'MAC000006', 'MAC000007', 'MAC000008', 'MAC000009', 'MAC000010', 'MAC000011', 'MAC000012', 'MAC000013', 'MAC000016', 'MAC000018', 'MAC000019', 'MAC000020', 'MAC000021', 'MAC000022', 'MAC000023', 'MAC000024', 'MAC000025', 'MAC000026', 'MAC000027', 'MAC000028', 'MAC000029', 'MAC000030', 'MAC000032', 'MAC000033', 'MAC000034', 'MAC000035', 'MAC000036'])

In [24]:
client_X_val.keys()

dict_keys(['MAC000002', 'MAC000003', 'MAC000004', 'MAC000006', 'MAC000007', 'MAC000008', 'MAC000009', 'MAC000010', 'MAC000011', 'MAC000012', 'MAC000013', 'MAC000016', 'MAC000018', 'MAC000019', 'MAC000020', 'MAC000021', 'MAC000022', 'MAC000023', 'MAC000024', 'MAC000025', 'MAC000026', 'MAC000027', 'MAC000028', 'MAC000029', 'MAC000030', 'MAC000032', 'MAC000033', 'MAC000034', 'MAC000035', 'MAC000036'])

In [25]:
for client in client_X_train:
    print(f"\nClient: {client}")
    print(f"X_train shape: {client_X_train[client].shape}, y_train shape: {client_y_train[client].shape}")
    print(f"X_val shape: {client_X_val[client].shape}, y_val shape: {client_y_val[client].shape}")


Client: MAC000002
X_train shape: (19317, 10, 1, 1), y_train shape: (19317, 1)
X_val shape: (4821, 10, 1, 1), y_val shape: (4821, 1)

Client: MAC000003
X_train shape: (28366, 10, 1, 1), y_train shape: (28366, 1)
X_val shape: (7083, 10, 1, 1), y_val shape: (7083, 1)

Client: MAC000004
X_train shape: (25332, 10, 1, 1), y_train shape: (25332, 1)
X_val shape: (6325, 10, 1, 1), y_val shape: (6325, 1)

Client: MAC000006
X_train shape: (29159, 10, 1, 1), y_train shape: (29159, 1)
X_val shape: (7282, 10, 1, 1), y_val shape: (7282, 1)

Client: MAC000007
X_train shape: (20027, 10, 1, 1), y_train shape: (20027, 1)
X_val shape: (4999, 10, 1, 1), y_val shape: (4999, 1)

Client: MAC000008
X_train shape: (20801, 10, 1, 1), y_train shape: (20801, 1)
X_val shape: (5192, 10, 1, 1), y_val shape: (5192, 1)

Client: MAC000009
X_train shape: (20181, 10, 1, 1), y_train shape: (20181, 1)
X_val shape: (5037, 10, 1, 1), y_val shape: (5037, 1)

Client: MAC000010
X_train shape: (20030, 10, 1, 1), y_train shape: (

In [26]:
def get_input_dims(X_train, exogenous_data_train):
    if args.model_name == "mlp":
        input_dim = X_train.shape[1] * X_train.shape[2]
    else:
        input_dim = X_train.shape[2]
    
    if exogenous_data_train is not None:
        if len(exogenous_data_train) == 1:
            cid = next(iter(exogenous_data_train.keys()))
            exogenous_dim = exogenous_data_train[cid].shape[1]
        else:
            exogenous_dim = exogenous_data_train["all"].shape[1]
    else:
        exogenous_dim = 0
    
    return input_dim, exogenous_dim

In [27]:
def get_model(model: str,
              input_dim: int,
              out_dim: int,
              lags: int = 10,
              exogenous_dim: int = 0,
              seed=0):
    if model == "mlp":
        model = MLP(input_dim=input_dim, layer_units=[256, 128, 64], num_outputs=out_dim)
    elif model == "rnn":
        model = RNN(input_dim=input_dim, rnn_hidden_size=128, num_rnn_layers=1, rnn_dropout=0.0,
                    layer_units=[128], num_outputs=out_dim, matrix_rep=True, exogenous_dim=exogenous_dim)
    elif model == "lstm":
        model = LSTM(input_dim=input_dim, lstm_hidden_size=128, num_lstm_layers=1, lstm_dropout=0.0,
                     layer_units=[128], num_outputs=out_dim, matrix_rep=True, exogenous_dim=exogenous_dim)
    elif model == "gru":
        model = GRU(input_dim=input_dim, gru_hidden_size=128, num_gru_layers=1, gru_dropout=0.0,
                    layer_units=[128], num_outputs=out_dim, matrix_rep=True, exogenous_dim=exogenous_dim)
    elif model == "cnn":
        model = CNN(num_features=input_dim, lags=lags, exogenous_dim=exogenous_dim, out_dim=out_dim)
    elif model == "da_encoder_decoder":
        model = DualAttentionAutoEncoder(input_dim=input_dim, architecture="lstm", matrix_rep=True)
    else:
        raise NotImplementedError("Specified model is not implemented. Plese define your own model or choose one from ['mlp', 'rnn', 'lstm', 'gru', 'cnn', 'da_encoder_decoder']")
    return model

In [28]:
# 定义模型
args.model_name = "cnn"  # 可以选择: "mlp", "rnn", "lstm", "gru", "cnn", "da_encoder_decoder"

input_dim, exogenous_dim = get_input_dims(X_train, exogenous_data_train)

print(f"输入维度: {input_dim}, 外生特征维度: {exogenous_dim}")
print(f"输出维度: {y_train.shape[1]}")  # 对于电力数据应该是1（只有kwh一个目标）

model = get_model(model=args.model_name,
                  input_dim=input_dim,
                  out_dim=y_train.shape[1],  # 电力数据只有1个目标（kwh）
                  lags=args.num_lags,
                  exogenous_dim=exogenous_dim,
                  seed=args.seed)

输入维度: 1, 外生特征维度: 0
输出维度: 1


In [29]:
model

CNN(
  (activation): ReLU()
  (conv1): Conv2d(1, 16, kernel_size=(16, 3), stride=(1, 1), padding=same)
  (conv2): Conv2d(16, 16, kernel_size=(3, 5), stride=(1, 1), padding=same)
  (conv3): Conv2d(16, 32, kernel_size=(4, 3), stride=(1, 1), padding=same)
  (conv4): Conv2d(32, 32, kernel_size=(4, 3), stride=(1, 1), padding=same)
  (pool): AvgPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0)
  (fc): Linear(in_features=160, out_features=1, bias=True)
)

### Fit函数启动每个客户本地模型的训练过程，然后在中央服务器上执行参数聚合，进行N轮指定的联邦学习轮数。

In [30]:
def fit(model, X_train, y_train, X_val, y_val, 
        exogenous_data_train=None, exogenous_data_val=None, 
        idxs=[0], # 目标在X中的索引，对于电力数据只有kwh一个目标，所以是[0]
        log_per=1,
        client_creation_fn = None, # client specification
        local_train_params=None, # local params
        aggregation_params=None, # aggregation params
        use_carbontracker=True
       ):
    # client creation definition
    if client_creation_fn is None:
        client_creation_fn = create_regression_client
    # local params
    if local_train_params is None:
        local_train_params = {
            "epochs": args.epochs, "optimizer": args.optimizer, "lr": args.lr,
            "criterion": args.criterion, "early_stopping": args.local_early_stopping,
            "patience": args.local_patience, "device": device
        }
    
    train_loaders, val_loaders = [], []
    
    # get data per client
    for client in X_train:
        if client == "all":
            continue
        if exogenous_data_train is not None:
            tmp_exogenous_data_train = exogenous_data_train[client]
            tmp_exogenous_data_val = exogenous_data_val[client]
        else:
            tmp_exogenous_data_train = None
            tmp_exogenous_data_val = None
    
        num_features = len(X_train[client][0][0])
        
        # to torch loader
        train_loaders.append(
            to_torch_dataset(
                X_train[client], y_train[client],
                num_lags=args.num_lags,
                num_features=num_features,
                exogenous_data=tmp_exogenous_data_train,
                indices=idxs,
                batch_size=args.batch_size,
                shuffle=False
            )
        )
        val_loaders.append(
            to_torch_dataset(
                X_val[client], y_val[client],
                num_lags=args.num_lags,
                num_features=num_features,
                exogenous_data=tmp_exogenous_data_val,
                indices=idxs,
                batch_size=args.batch_size,
                shuffle=False
            )
            
        )
        
    # create clients with their local data
    cids = [k for k in X_train.keys() if k != "all"]
    clients = [
        client_creation_fn(
            cid=cid, # client id
            model=model, # the global model
            train_loader=train_loader, # the local train loader
            test_loader=val_loader, # the local val loader
            local_params=local_train_params # local parameters
        )
        for cid, train_loader, val_loader in zip(cids, train_loaders, val_loaders)
    ]
    
    # represent clients to server
    client_proxies = [
        SimpleClientProxy(cid, client) for cid, client in zip(cids, clients)
    ]
    
    # represent the server
    server = Server(
        client_proxies=client_proxies, # the client representations
        aggregation=args.aggregation, # the aggregation algorithm
        aggregation_params=aggregation_params, # aggregation specific params
        local_params_fn=None, # we can change the local params on demand
    )
    # Note that the client manager instance will be initialized automatically. You can define your own client manager.

    # train with FL
    model_params, history = server.fit(args.fl_rounds, args.fraction, use_carbontracker=use_carbontracker)
    
    params_dict = zip(model.state_dict().keys(), model_params)
    state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
    model = copy.deepcopy(model)
    model.load_state_dict(state_dict, strict=True)
    
    return model, history

In [31]:
# federated local params
local_train_params = {"epochs": args.epochs, "optimizer": args.optimizer, "lr": args.lr,
                      "criterion": args.criterion, "early_stopping": args.local_early_stopping,
                      "patience": args.local_patience, "device": device
                      }

In [32]:
global_model, history = fit(
    model,
    client_X_train,
    client_y_train, 
    client_X_val, 
    client_y_val, 
    local_train_params=local_train_params
)

INFO logger 2026-01-07 12:26:16,815 | server.py:62 | Initializing client manager...
INFO logger 2026-01-07 12:26:16,816 | server.py:69 | Registering clients...
INFO logger 2026-01-07 12:26:16,816 | client_manager.py:66 | Registered client with id: MAC000002
INFO logger 2026-01-07 12:26:16,817 | client_manager.py:66 | Registered client with id: MAC000003
INFO logger 2026-01-07 12:26:16,817 | client_manager.py:66 | Registered client with id: MAC000004
INFO logger 2026-01-07 12:26:16,817 | client_manager.py:66 | Registered client with id: MAC000006
INFO logger 2026-01-07 12:26:16,818 | client_manager.py:66 | Registered client with id: MAC000007
INFO logger 2026-01-07 12:26:16,818 | client_manager.py:66 | Registered client with id: MAC000008
INFO logger 2026-01-07 12:26:16,818 | client_manager.py:66 | Registered client with id: MAC000009
INFO logger 2026-01-07 12:26:16,819 | client_manager.py:66 | Registered client with id: MAC000010
INFO logger 2026-01-07 12:26:16,819 | client_manager.py:

In [33]:
global_model.state_dict()

OrderedDict([('conv1.weight',
              tensor([[[[ 4.4131e-03,  1.4375e-02, -3.1274e-02],
                        [ 6.2170e-02,  6.4378e-02, -1.8940e-02],
                        [ 1.6599e-03,  5.2700e-02, -2.0501e-02],
                        [ 6.6007e-02,  1.0285e-01, -6.8485e-02],
                        [ 4.1137e-02,  1.0949e-01, -6.5779e-02],
                        [ 5.3469e-04,  8.8589e-02, -1.0161e-02],
                        [ 1.1508e-02,  1.5503e-01, -7.4601e-02],
                        [-6.3982e-02,  1.2996e-01,  4.6891e-02],
                        [-6.2225e-02,  1.6585e-01, -1.4613e-02],
                        [ 5.8084e-02,  4.0421e-02,  2.6937e-02],
                        [ 6.4973e-02,  5.5939e-02, -8.1512e-02],
                        [-2.0644e-02, -1.3782e-01,  1.8062e-02],
                        [ 1.0321e-03, -9.8871e-03, -3.0915e-02],
                        [-1.1754e-02,  1.0639e-01,  3.8454e-02],
                        [-1.0908e-02, -8.9882e-02, -3.3777e-

In [34]:
history


History (client, train losses):
	MAC000018: {1: 2.242149137219953e-05, 2: 2.2372613796554556e-05, 3: 2.2224041876070156e-05, 4: 2.2218340561062196e-05, 5: 2.218705187338855e-05, 6: 2.226969979926774e-05, 7: 2.226584119990558e-05, 8: 2.2218259496498152e-05, 9: 2.2232296165810002e-05, 10: 2.2292085039538504e-05}
	MAC000030: {1: 1.9109313310155993e-05, 2: 1.841936439718078e-05, 3: 1.867763907113196e-05, 4: 1.8438922628149155e-05, 5: 1.816412086705124e-05, 6: 1.805530053695257e-05, 7: 1.7871650860166695e-05, 8: 1.765793681054709e-05, 9: 1.7578908269418027e-05, 10: 1.7813908856814532e-05}
	MAC000019: {1: 3.497250234530492e-05, 2: 3.4820144689347955e-05, 3: 3.4607799325874996e-05, 4: 3.457063169179525e-05, 5: 3.462446815008326e-05, 6: 3.4487421748102415e-05, 7: 3.449160095970575e-05, 8: 3.4458185163029015e-05, 9: 3.443719159921671e-05, 10: 3.44563252767221e-05}
	MAC000003: {1: 8.4979413185582e-05, 2: 8.03457845901297e-05, 3: 7.423099446501987e-05, 4: 7.317886529037922e-05, 5: 6.744740323881